In [1]:
import sys
import os, certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

import os
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

ROOT = Path('.')
HW_DIR = ROOT
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
for p in [HW_DIR, ARTIFACTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULTS = {
    'dataset': 'CIFAR10',
    'seed': 42,
    'batch_size_cpu': 64,
    'batch_size_gpu': 256,
    'max_epochs': 30,
}


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13 -m pip install --upgrade pip


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def device_info():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def get_dataloaders(dataset_name: str, seed: int, batch_size: int, num_workers: int = 2):
    # Выбор трансформаций и класса датасета
    dataset_name = dataset_name.upper()
    if dataset_name == 'CIFAR10':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        ds_cls = torchvision.datasets.CIFAR10
    elif dataset_name == 'EMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        ds_cls = lambda root, train, download, transform: torchvision.datasets.EMNIST(
            root=root, split='balanced', train=train, download=download, transform=transform)
    else:  # по умолчанию KMNIST
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        ds_cls = torchvision.datasets.KMNIST

    # Путь к данным
    data_root = './data'
    
    # Загрузка train/test
    train_full = ds_cls(root=data_root, train=True, download=True, transform=transform)
    test_ds = ds_cls(root=data_root, train=False, download=True, transform=transform)

    # Разбиение на train/val с фиксированным seed
    val_ratio = 0.2
    val_size = int(len(train_full) * val_ratio)
    train_size = len(train_full) - val_size
    gen = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(train_full, [train_size, val_size], generator=gen)

    # Настройка DataLoader
    device = device_info()
    num_workers = 0 if os.name == 'nt' else num_workers
    pin_memory = (device.type == 'cuda')

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=pin_memory)

    # Sanity check
    x, y = next(iter(train_loader))
    print(f"Batch shapes: x={x.shape}, y={y.shape}")
    print(f"x range: [{x.min().item()}, {x.max().item()}], y range: [{y.min().item()}, {y.max().item()}]")

    return train_loader, val_loader, test_loader


In [4]:
class MLP(nn.Module):
    def __init__(self, in_features: int, hidden_sizes=(512, 256), n_classes=10, activation='relu', dropout_p: Optional[float]=None, use_batchnorm=False):
        super().__init__()
        layers = []
        act = nn.ReLU if activation == 'relu' else nn.Tanh
        prev = in_features
        for i, h in enumerate(hidden_sizes):
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act())
            if dropout_p is not None and dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x expected shape (B, C, H, W) -> flatten
        if x.dim() > 2:
            x = torch.flatten(x, start_dim=1)
        return self.net(x)


In [5]:
def accuracy_from_logits(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y_true).float().mean().item()


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_acc += accuracy_from_logits(logits, yb) * batch_size
        n += batch_size
    return total_loss / n, total_acc / n


In [6]:
def evaluate(model: nn.Module, loader: DataLoader, criterion, device) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_acc += accuracy_from_logits(logits, yb) * batch_size
            n += batch_size
    return total_loss / n, total_acc / n

In [7]:
class EarlyStopping:
    def __init__(self, patience=3, mode='max'):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad = 0

    def step(self, value):
        if self.best is None:
            self.best = value
            self.num_bad = 0
            return False
        improve = (value > self.best) if self.mode == 'max' else (value < self.best)
        if improve:
            self.best = value
            self.num_bad = 0
            return False
        else:
            self.num_bad += 1
            return self.num_bad >= self.patience

In [8]:
@dataclass
class RunResult:
    experiment_id: str
    dataset: str
    seed: int
    model_summary: str
    optimizer: str
    lr: float
    momentum: Optional[float]
    weight_decay: float
    epochs_trained: int
    best_val_accuracy: float
    best_val_loss: float

In [9]:
def run_experiment(exp_id: str, cfg: Dict[str, Any], save_best: bool = False, early_stopping: Optional[EarlyStopping] = None):
    import time, traceback

    start_run = time.time()
    try:
        # cfg contains dataset, seed, model params, optimizer params, epochs
        print(f"[{exp_id}] run_experiment START", flush=True)
        set_seed(cfg['seed'])

        device = device_info()
        batch_size = cfg.get('batch_size') or (DEFAULTS['batch_size_gpu'] if device.type == 'cuda' else DEFAULTS['batch_size_cpu'])
        print(f"[{exp_id}] device={device}, batch_size={batch_size}", flush=True)

        # Force single-process DataLoader while debugging (common hang source)
        try:
            train_loader, val_loader, test_loader = get_dataloaders(cfg['dataset'], cfg['seed'], batch_size, num_workers=0)
        except TypeError:
            # Fallback if get_dataloaders doesn't accept num_workers kwarg
            train_loader, val_loader, test_loader = get_dataloaders(cfg['dataset'], cfg['seed'], batch_size)
        print(f"[{exp_id}] dataloaders obtained: train_batches={len(train_loader)}, val_batches={len(val_loader)}, test_batches={len(test_loader)}", flush=True)

        # Quick single-batch smoke test to ensure DataLoader yields and shapes are OK
        try:
            sample_iter = iter(train_loader)
            sample_x, sample_y = next(sample_iter)
            print(f"[{exp_id}] sample batch shapes: x={sample_x.shape}, y={sample_y.shape}, x_range=[{float(sample_x.min()):.4f},{float(sample_x.max()):.4f}]", flush=True)
        except Exception:
            print(f"[{exp_id}] WARN: failed to iterate single batch from train_loader — traceback follows", flush=True)
            traceback.print_exc()
            # don't early-return; keep trying (useful to see later prints)

        in_features = int(np.prod(sample_x.shape[1:]))
        # preserve original n_classes resolution (keep your existing logic)
        if hasattr(train_loader.dataset, 'dataset'):
            try:
                n_classes = len(torch.unique(torch.tensor([y for _, y in train_loader.dataset])))
            except Exception:
                n_classes = cfg.get('n_classes', 10)
        else:
            n_classes = cfg.get('n_classes', 10)

        model = MLP(
            in_features=in_features,
            hidden_sizes=cfg['hidden_sizes'],
            n_classes=cfg.get('n_classes', n_classes),
            activation=cfg.get('activation','relu'),
            dropout_p=cfg.get('dropout', None),
            use_batchnorm=cfg.get('batchnorm', False)
        )
        model = model.to(device)
        print(f"[{exp_id}] model instantiated and moved to device", flush=True)

        criterion = nn.CrossEntropyLoss()

        opt_name = cfg.get('optimizer','adam').lower()
        if opt_name == 'sgd':
            optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=cfg.get('momentum',0.0), weight_decay=cfg.get('weight_decay',0.0))
        else:
            optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('weight_decay',0.0))
        print(f"[{exp_id}] optimizer: {opt_name}, lr={cfg.get('lr')}", flush=True)

        max_epochs = cfg.get('epochs', DEFAULTS.get('max_epochs', 20))
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

        best_val_acc = 0.0
        best_val_loss = float('inf')
        best_state = None
        stop = False
        epoch = 0

        for epoch in range(1, max_epochs+1):
            epoch_start = time.time()
            print(f"[{exp_id}] === epoch {epoch}/{max_epochs} START ===", flush=True)

            # train_one_epoch may hide internals — we at least print before/after
            try:
                train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            except Exception:
                print(f"[{exp_id}] ERROR during train_one_epoch — traceback follows", flush=True)
                traceback.print_exc()
                raise

            # synchronize CUDA to detect hangs / async errors early (no-op on CPU)
            if device.type == 'cuda':
                try:
                    torch.cuda.synchronize()
                except Exception:
                    print(f"[{exp_id}] WARNING: cuda.synchronize() failed", flush=True)
                    traceback.print_exc()

            try:
                val_loss, val_acc = evaluate(model, val_loader, criterion, device)
            except Exception:
                print(f"[{exp_id}] ERROR during evaluate (validation) — traceback follows", flush=True)
                traceback.print_exc()
                raise

            if device.type == 'cuda':
                try:
                    torch.cuda.synchronize()
                except Exception:
                    pass

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)

            print(f"[{exp_id}] epoch {epoch} results: train_loss={train_loss:.4f} val_loss={val_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f} (epoch_time={time.time()-epoch_start:.2f}s)", flush=True)

            # save-best logic: compare to previous best (capture previous value first)
            prev_best_val_acc = best_val_acc
            prev_best_val_loss = best_val_loss

            if val_acc > best_val_acc:
                best_val_acc = val_acc
            if val_loss < best_val_loss:
                best_val_loss = val_loss

            if save_best and val_acc > prev_best_val_acc:
                best_state = model.state_dict()
                print(f"[{exp_id}] New best_val_acc={best_val_acc:.4f} -> saving best_state (in-memory).", flush=True)

            if early_stopping is not None:
                try:
                    if early_stopping.step(val_acc):
                        print(f"[{exp_id}] EarlyStopping triggered at epoch {epoch}.", flush=True)
                        stop = True
                        break
                except Exception:
                    print(f"[{exp_id}] WARNING: early_stopping.step raised an exception — traceback follows", flush=True)
                    traceback.print_exc()

        epochs_trained = epoch
        print(f"[{exp_id}] training loop finished, epochs_trained={epochs_trained}, elapsed={time.time()-start_run:.2f}s", flush=True)

        # final test evaluation
        try:
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)
            print(f"[{exp_id}] test results: loss={test_loss:.4f}, acc={test_acc:.4f}", flush=True)
        except Exception:
            print(f"[{exp_id}] ERROR during evaluate (test) — traceback follows", flush=True)
            traceback.print_exc()
            test_loss, test_acc = float('inf'), 0.0

        # persist best model & artifacts if requested
        if save_best and best_state is not None:
            try:
                torch.save(best_state, ARTIFACTS / 'best_model.pt')
                with open(ARTIFACTS / 'best_config.json', 'w') as f:
                    json.dump(cfg, f, indent=2)
                plt.figure()
                plt.plot(np.arange(1, len(history['train_loss'])+1), history['train_loss'], label='train_loss')
                plt.plot(np.arange(1, len(history['val_loss'])+1), history['val_loss'], label='val_loss')
                plt.legend(); plt.grid(); plt.xlabel('epoch'); plt.title(f'{exp_id} loss'); plt.savefig(FIGURES / 'curves_best.png'); plt.close()
                print(f"[{exp_id}] saved best model and curves to ARTIFACTS/Figures", flush=True)
            except Exception:
                print(f"[{exp_id}] WARNING: saving artifacts failed — traceback follows", flush=True)
                traceback.print_exc()

        result = RunResult(
            experiment_id=exp_id,
            dataset=cfg['dataset'],
            seed=cfg['seed'],
            model_summary=str({'hidden': cfg['hidden_sizes'], 'dropout': cfg.get('dropout'), 'batchnorm': cfg.get('batchnorm')}),
            optimizer=cfg.get('optimizer'),
            lr=cfg.get('lr'),
            momentum=cfg.get('momentum', 0.0),
            weight_decay=cfg.get('weight_decay', 0.0),
            epochs_trained=epochs_trained,
            best_val_accuracy=best_val_acc,
            best_val_loss=best_val_loss,
        )
        print(f"[{exp_id}] run_experiment END (total elapsed: {time.time()-start_run:.2f}s)", flush=True)
        return result, history, (test_loss, test_acc)

    except Exception as e:
        print(f"[{exp_id}] UNCAUGHT exception in run_experiment: {e}", flush=True)
        traceback.print_exc()
        # Re-raise so caller knows experiment failed (you can change to return partials if desired)
        raise

In [10]:
def main():
    dataset = DEFAULTS['dataset']
    seed = DEFAULTS['seed']
    device = device_info()
    batch_size = DEFAULTS['batch_size_gpu'] if device.type == 'cuda' else DEFAULTS['batch_size_cpu']

    base_cfg = {
        'dataset': dataset,
        'seed': seed,
        'hidden_sizes': [512, 256],
        'n_classes': 10,
        'activation': 'relu',
        'batchnorm': False,
        'dropout': None,
        'optimizer': 'adam',
        'lr': 1e-3,
        'momentum': 0.0,
        'weight_decay': 0.0,
        'epochs': 20,
        'batch_size': batch_size,
    }

    runs = []

    # E1: base
    cfg = dict(base_cfg)
    cfg.update({'dropout': None, 'batchnorm': False})
    r1, h1, t1 = run_experiment('E1', cfg, save_best=False, early_stopping=None)
    runs.append(asdict(r1))

    # E2: Dropout
    cfg = dict(base_cfg)
    cfg.update({'dropout': 0.3, 'batchnorm': False})
    r2, h2, t2 = run_experiment('E2', cfg, save_best=False, early_stopping=None)
    runs.append(asdict(r2))

    # E3: BatchNorm
    cfg = dict(base_cfg)
    cfg.update({'dropout': None, 'batchnorm': True})
    r3, h3, t3 = run_experiment('E3', cfg, save_best=False, early_stopping=None)
    runs.append(asdict(r3))

    # choose best of E2/E3 by val_acc -> E4 with early stopping
    best_of_23 = max([(r2.best_val_accuracy, r2), (r3.best_val_accuracy, r3)], key=lambda x: x[0])[1]
    # reconstruct cfg for best
    cfg = dict(base_cfg)
    cfg.update({'dropout': 0.3 if 'dropout' in r2.model_summary and '0.3' in r2.model_summary else None, 'batchnorm': True if 'True' in r3.model_summary else False})
    # Better: pick E2 if r2.best_val_accuracy >= r3.best_val_accuracy else E3
    cfg = dict(base_cfg)
    if r2.best_val_accuracy >= r3.best_val_accuracy:
        cfg.update({'dropout': 0.3, 'batchnorm': False})
    else:
        cfg.update({'dropout': None, 'batchnorm': True})
    # E4 with EarlyStopping
    r4, h4, t4 = run_experiment('E4', cfg, save_best=True, early_stopping=EarlyStopping(patience=4, mode='max'))
    runs.append(asdict(r4))

    # Part B: O1-O3 on E4 architecture
    arch_cfg = dict(cfg)
    # O1: LR too large
    o1_cfg = dict(arch_cfg); o1_cfg.update({'optimizer':'adam','lr':1e-1,'epochs':8})
    o1, h_o1, t_o1 = run_experiment('O1', o1_cfg, save_best=False)
    runs.append(asdict(o1))
    # O2: LR too small
    o2_cfg = dict(arch_cfg); o2_cfg.update({'optimizer':'adam','lr':1e-5,'epochs':8})
    o2, h_o2, t_o2 = run_experiment('O2', o2_cfg, save_best=False)
    runs.append(asdict(o2))
    # O3: SGD+momentum + weight decay
    o3_cfg = dict(arch_cfg); o3_cfg.update({'optimizer':'sgd','lr':1e-2,'momentum':0.9,'weight_decay':1e-4,'epochs':15})
    o3, h_o3, t_o3 = run_experiment('O3', o3_cfg, save_best=False)
    runs.append(asdict(o3))

    # Save runs.csv
    import csv
    keys = ['experiment_id','dataset','seed','model_summary','optimizer','lr','momentum','weight_decay','epochs_trained','best_val_accuracy','best_val_loss']
    with open(ARTIFACTS / 'runs.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for r in runs:
            writer.writerow({k: r.get(k) for k in keys})

    plt.figure(figsize=(8,4))
    plt.plot(np.arange(1,len(h_o1['train_loss'])+1), h_o1['train_loss'], label='O1 train_loss')
    plt.plot(np.arange(1,len(h_o1['val_loss'])+1), h_o1['val_loss'], label='O1 val_loss')
    plt.plot(np.arange(1,len(h_o2['train_loss'])+1), h_o2['train_loss'], '--', label='O2 train_loss')
    plt.plot(np.arange(1,len(h_o2['val_loss'])+1), h_o2['val_loss'], '--', label='O2 val_loss')
    plt.legend(); plt.grid(); plt.xlabel('epoch'); plt.title('LR extremes (O1 vs O2)'); plt.savefig(FIGURES / 'curves_lr_extremes.png'); plt.close()


    
if __name__ == '__main__':
    main()


[E1] run_experiment START
[E1] device=cpu, batch_size=64
Batch shapes: x=torch.Size([64, 3, 32, 32]), y=torch.Size([64])
x range: [-1.0, 1.0], y range: [0, 9]
[E1] dataloaders obtained: train_batches=625, val_batches=157, test_batches=157
[E1] sample batch shapes: x=torch.Size([64, 3, 32, 32]), y=torch.Size([64]), x_range=[-1.0000,1.0000]
[E1] model instantiated and moved to device
[E1] optimizer: adam, lr=0.001
[E1] === epoch 1/20 START ===
[E1] epoch 1 results: train_loss=1.6636 val_loss=1.5399 train_acc=0.4086 val_acc=0.4603 (epoch_time=10.04s)
[E1] === epoch 2/20 START ===
[E1] epoch 2 results: train_loss=1.4519 val_loss=1.4296 train_acc=0.4853 val_acc=0.4985 (epoch_time=10.71s)
[E1] === epoch 3/20 START ===
[E1] epoch 3 results: train_loss=1.3400 val_loss=1.4260 train_acc=0.5258 val_acc=0.4967 (epoch_time=9.11s)
[E1] === epoch 4/20 START ===
[E1] epoch 4 results: train_loss=1.2399 val_loss=1.4035 train_acc=0.5613 val_acc=0.5179 (epoch_time=8.78s)
[E1] === epoch 5/20 START ===
[E1]